In [ ]:
!pip install scikit-learn

In [ ]:
# ============================================================================
# FER-2013 Facial Emotion Recognition — Kaggle Benchmark Task
# ============================================================================
# Task: Given a 48×48 grayscale facial image from FER-2013,
#        predict the emotion being expressed. Image input ONLY.
# Input: Single .jpg image (no text, audio, or video)
# Output: One of 7 emotion labels
# Metric: Weighted-Average F1 Score (float, 0.0 – 1.0)
# Samples: 5 (stratified across emotion classes, deterministic)
# ============================================================================

import kaggle_benchmarks as kbench
from kaggle_benchmarks.content_types import images
from dataclasses import dataclass
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import os
import re

# ============================================================================
# CONFIGURATION
# ============================================================================

DATASET_ROOT = "/kaggle/input/datasets/msambare/fer2013"
NUM_SAMPLES = 250
RANDOM_SEED = 42

VALID_EMOTIONS = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

EMOTION_SYNONYMS = {
    "anger": "angry", "mad": "angry", "furious": "angry",
    "irritated": "angry", "enraged": "angry",
    "disgusted": "disgust", "disgusting": "disgust",
    "repulsed": "disgust", "revulsion": "disgust",
    "fearful": "fear", "afraid": "fear", "scared": "fear",
    "frightened": "fear", "anxious": "fear", "terrified": "fear",
    "happiness": "happy", "joy": "happy", "joyful": "happy",
    "cheerful": "happy", "excited": "happy", "smiling": "happy",
    "calm": "neutral", "normal": "neutral", "none": "neutral", "flat": "neutral",
    "sadness": "sad", "sorrowful": "sad", "unhappy": "sad",
    "melancholy": "sad", "depressed": "sad",
    "surprised": "surprise", "shocked": "surprise",
    "astonished": "surprise", "startled": "surprise",
}


# ============================================================================
# STRUCTURED OUTPUT SCHEMA
# ============================================================================

@dataclass
class EmotionPrediction:
    """The LLM must return exactly one emotion label."""
    emotion: str


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def discover_test_images(dataset_root):
    """Walk the test/ directory to find all .jpg images organized by emotion subfolder."""
    test_dir = None
    candidates = [
        os.path.join(dataset_root, "test"),
        os.path.join(dataset_root, "FER-2013", "test"),
        os.path.join(dataset_root, "fer2013", "test"),
    ]

    for candidate in candidates:
        if os.path.isdir(candidate):
            test_dir = candidate
            break

    if test_dir is None:
        for dirpath, dirnames, filenames in os.walk(dataset_root):
            subdir_names = set(d.lower() for d in dirnames)
            if len(subdir_names.intersection(set(VALID_EMOTIONS))) >= 5:
                test_dir = dirpath
                break

    if test_dir is None:
        raise FileNotFoundError(
            f"Could not find test directory with emotion subfolders under {dataset_root}. "
            f"Expected structure: test/angry/, test/happy/, etc."
        )

    print(f"[INFO] Test directory: {test_dir}")

    all_images = []
    for emotion_folder in sorted(os.listdir(test_dir)):
        emotion_path = os.path.join(test_dir, emotion_folder)
        if not os.path.isdir(emotion_path):
            continue

        emotion_label = emotion_folder.strip().lower()
        if emotion_label not in VALID_EMOTIONS:
            print(f"[WARN] Skipping unrecognized folder: {emotion_folder}")
            continue

        for fname in sorted(os.listdir(emotion_path)):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            if fname.startswith("._"):
                continue

            all_images.append({
                "filepath": os.path.join(emotion_path, fname),
                "emotion": emotion_label,
                "filename": fname,
            })

    print(f"[INFO] Discovered {len(all_images)} test images across emotions:")
    emotion_counts = {}
    for img in all_images:
        emotion_counts[img["emotion"]] = emotion_counts.get(img["emotion"], 0) + 1
    for emo in VALID_EMOTIONS:
        print(f"  {emo}: {emotion_counts.get(emo, 0)}")

    return all_images


def select_stratified_samples(all_images, n=NUM_SAMPLES):
    """Select N samples via stratified sampling across emotions."""
    df = pd.DataFrame(all_images)

    if len(df) == 0:
        raise ValueError("No valid test images found.")

    actual_n = min(n, len(df))
    present_emotions = [e for e in VALID_EMOTIONS if e in df["emotion"].values]

    if actual_n < len(present_emotions):
        class_counts = df["emotion"].value_counts()
        priority_emotions = class_counts.index.tolist()[:actual_n]

        guaranteed_parts = []
        for emo in priority_emotions:
            emo_pool = df[df["emotion"] == emo]
            sampled = emo_pool.sample(n=1, random_state=RANDOM_SEED)
            guaranteed_parts.append(sampled)

        selected_df = pd.concat(guaranteed_parts)
    else:
        min_per_class = 1
        guaranteed_parts = []
        remaining_budget = actual_n

        for emo in present_emotions:
            emo_pool = df[df["emotion"] == emo]
            take = min(min_per_class, len(emo_pool))
            sampled = emo_pool.sample(n=take, random_state=RANDOM_SEED)
            guaranteed_parts.append(sampled)
            remaining_budget -= take

        guaranteed_df = pd.concat(guaranteed_parts)
        guaranteed_indices = set(guaranteed_df.index)

        if remaining_budget > 0:
            leftover_pool = df[~df.index.isin(guaranteed_indices)]
            take_more = min(remaining_budget, len(leftover_pool))

            if take_more > 0:
                proportional_parts = []
                for emo, grp in leftover_pool.groupby("emotion"):
                    take = min(
                        len(grp),
                        max(1, int(round(take_more * len(grp) / len(leftover_pool))))
                    )
                    proportional_parts.append(
                        grp.sample(n=take, random_state=RANDOM_SEED)
                    )
                proportional = pd.concat(proportional_parts)

                if len(proportional) > take_more:
                    proportional = proportional.sample(
                        n=take_more, random_state=RANDOM_SEED
                    )

                selected_df = pd.concat([guaranteed_df, proportional])
            else:
                selected_df = guaranteed_df
        else:
            selected_df = guaranteed_df

    selected_df = selected_df.sort_values("filepath").reset_index(drop=True)

    print(f"\n[INFO] Final selection: {len(selected_df)} samples")
    print(selected_df["emotion"].value_counts().to_string())
    return selected_df


def normalize_emotion(raw_prediction):
    """Normalize the LLM's predicted emotion to a canonical FER-2013 label."""
    cleaned = raw_prediction.strip().lower()
    cleaned = re.sub(r"[^a-z]", "", cleaned)

    if cleaned in VALID_EMOTIONS:
        return cleaned

    if cleaned in EMOTION_SYNONYMS:
        return EMOTION_SYNONYMS[cleaned]

    for emo in VALID_EMOTIONS:
        if emo in cleaned:
            return emo

    return "INVALID"


def build_prompt():
    """Build the classification prompt."""
    return (
        "You are an expert at recognizing human emotions from facial expressions.\n\n"
        "Look at this grayscale image of a person's face. Based on their "
        "facial expression, classify the emotion being expressed.\n\n"
        "Choose exactly one of the following 7 emotion categories:\n"
        "angry, disgust, fear, happy, neutral, sad, surprise\n\n"
        "Respond with a single JSON object containing one field 'emotion' "
        "with your chosen label."
    )


# ============================================================================
# MAIN BENCHMARK TASK
# ============================================================================

@kbench.task(
    name="fer_facial_emotion_recognition",
    description=(
        "Facial emotion recognition on the FER-2013 dataset. "
        "The model receives a single 48x48 grayscale face image and must classify "
        "the facial expression into one of 7 categories: "
        "angry, disgust, fear, happy, neutral, sad, surprise."
    ),
)
def fer_facial_emotion_recognition(llm) -> float:
    """
    Evaluate an LLM's ability to recognize emotions from facial images.
    Returns Weighted-Average F1 score between 0.0 and 1.0.
    """
    # STEP 1: Discover test images
    all_images = discover_test_images(DATASET_ROOT)

    # STEP 2: Select stratified samples
    samples = select_stratified_samples(all_images, n=NUM_SAMPLES)

    # STEP 3: Run predictions
    y_true = []
    y_pred = []
    results_log = []
    prompt_text = build_prompt()
    total = len(samples)

    for i, (idx, row) in enumerate(samples.iterrows()):
        ground_truth = row["emotion"]
        filepath = row["filepath"]
        fname = row["filename"]

        img = images.from_path(filepath)

        with kbench.chats.new(f"sample_{i}"):
            try:
                prediction = llm.prompt(
                    [prompt_text, img],
                    schema=EmotionPrediction,
                )
                raw_emotion = prediction.emotion

            except Exception as e_schema:
                print(f"  [WARN] Image+schema failed ({type(e_schema).__name__}): "
                      f"{e_schema}")
                print(f"  [FALLBACK] Trying image without schema...")

                with kbench.chats.new(f"sample_{i}_retry"):
                    raw_response = llm.prompt([prompt_text, img])
                    raw_emotion = raw_response.strip()

        predicted_emotion = normalize_emotion(raw_emotion)

        y_true.append(ground_truth)
        y_pred.append(predicted_emotion)
        results_log.append({
            "sample": i + 1,
            "filename": fname,
            "ground_truth": ground_truth,
            "predicted": predicted_emotion,
            "raw_output": str(raw_emotion)[:80],
            "correct": predicted_emotion == ground_truth,
        })

    # STEP 4: Compute Weighted-Average F1 Score
    all_labels = VALID_EMOTIONS + (["INVALID"] if "INVALID" in y_pred else [])

    waf1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        labels=all_labels,
        zero_division=0,
    )

    # STEP 5: Print summary
    print("\n" + "=" * 70)
    print("RESULTS SUMMARY")
    print("=" * 70)

    results_df = pd.DataFrame(results_log)
    print(results_df[["sample", "ground_truth", "predicted", "correct"]].to_string(
        index=False
    ))

    correct_count = sum(r["correct"] for r in results_log)
    invalid_count = sum(1 for p in y_pred if p == "INVALID")

    print(f"\nAccuracy:  {correct_count}/{total} ({correct_count/total:.1%})")
    print(f"Invalid:   {invalid_count}/{total}")
    print(f"WAF1:      {waf1:.4f}")

    if total >= 7:
        print(f"\nPer-class classification report:")
        print(classification_report(
            y_true, y_pred, labels=VALID_EMOTIONS, zero_division=0
        ))

    print("=" * 70)

    # STEP 6: Assertions — required for Kaggle to recognize the task
    kbench.assertions.assert_true(
        0.0 <= waf1 <= 1.0,
        expectation="Weighted-Average F1 score should be between 0 and 1."
    )

    kbench.assertions.assert_true(
        invalid_count < total,
        expectation="At least one prediction should be a valid emotion label."
    )

    kbench.assertions.assert_true(
        correct_count > 0,
        expectation="Model should correctly classify at least one sample."
    )

    return waf1


# ============================================================================
# RUN THE TASK
# ============================================================================

fer_facial_emotion_recognition.run(kbench.llm)

In [ ]:
%choose fer_facial_emotion_recognition